In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
file_path = "/Volumes/biodatabrickshybrid/default/biodatabrickshybrid/fastq/SRR16356247_1.fast"

lines_df = spark.read.text(file_path)

lines_df.show(5, truncate=False)

print("Liczba linii:", lines_df.count())
print("Liczba odczytów:", lines_df.count() // 4)

+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|@SRR16356247.1 1 length=151                                                                                                                            |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|
|+SRR16356247.1 1 length=151                                                                                                                            |
|A=AFFFFFFFFFFFFFF#F/FAFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF#FFFFFFFFFF#FFFFF

In [0]:
window = Window.orderBy(monotonically_increasing_id())

lines_with_id = lines_df.withColumn(
    "line_number",
    row_number().over(window) - 1
)

lines_with_id.show(8, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|value                                                                                                                                                  |line_number|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |
|+SRR16356247.1 1 length=151                                                                                                                            |2          |
|A=A

In [0]:
lines_typed = lines_with_id.withColumn(
    "line_type",
    when(col("line_number") % 4 == 0, "header")
    .when(col("line_number") % 4 == 1, "sequence")
    .when(col("line_number") % 4 == 2, "separator")
    .when(col("line_number") % 4 == 3, "quality")
)

lines_typed.show(12, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|value                                                                                                                                                  |line_number|line_type|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |
|+SRR16356247.1 1 length=151                                                                                            

In [0]:
lines_with_record = lines_typed.withColumn(
    "record_id",
    (col("line_number") / 4).cast("integer")
)

lines_with_record.show(16, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|value                                                                                                                                                  |line_number|line_type|record_id|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |0        |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |0        |
|+SRR16356247.1 1 length=151                                          

In [0]:
fastq_df = lines_with_record.groupBy("record_id").agg(
    first(
        when(col("line_type") == "header", col("value")),
        ignorenulls=True
    ).alias("header"),

    first(
        when(col("line_type") == "sequence", col("value")),
        ignorenulls=True
    ).alias("sequence"),

    first(
        when(col("line_type") == "separator", col("value")),
        ignorenulls=True
    ).alias("separator"),

    first(
        when(col("line_type") == "quality", col("value")),
        ignorenulls=True
    ).alias("quality")
)

fastq_df.show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|header                     |sequence                                                                                                                                               |separator                  |quality                                                                                                                                                |
+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------

In [0]:
validated_df = fastq_df.withColumn(
    "declared_length",
    regexp_extract(
        col("header"),
        r"length=(\d+)",
        1
    ).cast("integer")
).withColumn(
    "actual_length",
    length(col("sequence"))
)

validated_df.show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+-------------+
|record_id|header                     |sequence                                                                                                                                               |separator                  |quality                                                                                                                                                |declared_length|actual_length|
+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----

In [0]:
invalid_reads = validated_df.filter(
    col("declared_length") != col("actual_length")
)

invalid_reads.show()

print("Liczba niezgodnych rekordów:", invalid_reads.count())


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+------+--------+---------+-------+---------------+-------------+
|record_id|header|sequence|separator|quality|declared_length|actual_length|
+---------+------+--------+---------+-------+---------------+-------------+
+---------+------+--------+---------+-------+---------------+-------------+

Liczba niezgodnych rekordów: 0


**Zadanie 1 Walidacja integralności danych**

**Ile Jobs zostało utworzonych?**

W trakcie wykonania zadania zostały utworzone 2 Jobs. Pierwszy Job został uruchomiony przez akcję show(), natomiast drugi przez akcję count(). Widać, że każda akcja w Apache Spark powoduje utworzenie nowego Joba.

**Które operacje były transformacjami, a które akcjami?**

Transformacje:

withColumn()
regexp_extract()
length()
filter()


Akcje:

show()
count()
**
Ile Stage tworzy plan wykonania?**

Analiza Spark UI wykazała, że plan wykonania składał się z jednego Stage, ponieważ w analizowanym zapytaniu nie występowały operacje wymagające przetasowania danych (shuffle), takie jak groupBy() lub join().

**Ile Tasków wykonał Stage?**

Jedyny Stage wykonał 9 Tasków. Liczba Tasków odpowiada liczbie partycji DataFrame. 

Wnioski

Przeprowadzona walidacja potwierdziła poprawność danych FASTQ. Wszystkie rekordy posiadały zgodną deklarowaną i rzeczywistą długość sekwencji. Analiza Spark UI pokazała również zależność pomiędzy akcjami i Jobami oraz potwierdziła, że liczba Tasków odpowiada liczbie partycji przetwarzanych danych.

In [0]:
from pyspark.sql.functions import col

low_quality_reads = fastq_df.filter(
    col("quality").contains("#")
)

low_quality_reads.show(5, truncate=False)

print("Liczba odczytów zawierających znak '#':", low_quality_reads.count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|header                     |sequence                                                                                                                                               |separator                  |quality                                                                                                                                                |
+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------

**Zadanie 2 Wstępna analiza jakości**

**Porównajmy czas wykonania tego zadania z Zadaniem 1. Które było szybsze i dlaczego?**

Zadanie 2 wykonało się szybciej niż Zadanie 1, ponieważ wymagało jedynie sprawdzenia, czy w kolumnie quality występuje znak # za pomocą funkcji contains(). W Zadaniu 1 Spark musiał dodatkowo wykonać wyrażenie regularne (regexp_extract), obliczyć długość sekwencji (length) i porównać dwie wartości, co wymagało większej liczby operacji.

**Czy Spark wykorzystał cache?**

Na podstawie Query Profile stwierdzono, że Spark nie wykorzystał pamięci podręcznej (cache). Potwierdza to wartość "Bytes read from cache: 0%", co oznacza, że dane zostały ponownie odczytane i przetworzone podczas wykonywania zadania.

**Jaki był Locality Level tasków?**

W używanym środowisku Databricks Serverless informacja o Locality Level nie jest wyświetlana w Query Profile, dlatego nie było możliwości jej odczytania. Najbardziej korzystnym poziomem lokalności jest PROCESS_LOCAL, ponieważ dane są przetwarzane bezpośrednio w pamięci procesu wykonującego zadanie, co zapewnia najwyższą wydajność.

**Ile rekordów (Records) przetworzył każdy Task? Czy liczba ta jest zgodna z Twoimi
oczekiwaniami dotyczącymi podziału danych na partycje?**

Na podstawie Query Profile Spark odczytał 9 278 504 rekordy. Zadanie zostało wykonane z wykorzystaniem 9 Tasków, co oznacza, że dane zostały podzielone na 9 partycji. Każdy Task przetwarzał jedną partycję danych, co jest zgodne z architekturą Apache Spark.


In [0]:
from pyspark.sql.functions import col, length

analysis_df = (
    fastq_df
    .withColumn("has_low_quality", col("quality").contains("#"))
    .withColumn("seq_length", length(col("sequence")))
    .groupBy("has_low_quality", "seq_length")
    .count()
    .orderBy("has_low_quality", "seq_length")
)

analysis_df.count()
analysis_df.show()
analysis_df.select("seq_length").distinct().count()
analysis_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
from pyspark.sql.functions import col, length

analysis_df = (
    fastq_df
    .withColumn(
        "has_low_quality",
        col("quality").contains("#")
    )
    .withColumn(
        "seq_length",
        length(col("sequence"))
    )
    .groupBy(
        "has_low_quality",
        "seq_length"
    )
    .count()
    .orderBy(
        "has_low_quality",
        "seq_length"
    )

)



print("Liczba rekordów:")
print(analysis_df.count())


print("\nWyniki analizy:")
analysis_df.show(20, truncate=False)


print("\nLiczba różnych długości sekwencji:")
print(analysis_df.select("seq_length").distinct().count())


print("\nPonowne wyświetlenie wyników:")
analysis_df.show(20, truncate=False)



Liczba rekordów:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


2

Wyniki analizy:
+---------------+----------+-------+
|has_low_quality|seq_length|count  |
+---------------+----------+-------+
|false          |151       |2313942|
|true           |151       |5684   |
+---------------+----------+-------+


Liczba różnych długości sekwencji:
1

Ponowne wyświetlenie wyników:
+---------------+----------+-------+
|has_low_quality|seq_length|count  |
+---------------+----------+-------+
|false          |151       |2313942|
|true           |151       |5684   |
+---------------+----------+-------+



**Zadanie 3 Kompleksowa analiza i optymalizacja**

**Ile Jobs zostało utworzonych?**

Zostały utworzone 4 Joby, ponieważ wykonano cztery akcje: count(), show(), distinct().count() oraz ponownie show(). W Apache Spark każda akcja uruchamia nowy Job.
**
Stage 1**

W pierwszym etapie wykonywane były transformacje (withColumn, groupBy) oraz przygotowanie danych do agregacji. W środowisku Spark odpowiada to operacji Shuffle Write.

**Stage 2**

Drugi etap wykonywał agregację (count()) i zwracał wynik końcowy. W tym etapie wykonywany jest Shuffle Read.
**
Czy wykorzystano cache?**

Nie. Środowisko Databricks Serverless nie obsługuje funkcji cache(), a próba uruchomienia klastra Personal Compute zakończyła się błędem z powodu braku dostępnych zasobów (Estimated available: 0, requested: 4). Dlatego nie było możliwości sprawdzenia działania cache.

**Input Size i efektywność cache**

Ponieważ cache nie został wykorzystany, każda akcja ponownie odczytywała dane i wykonywała wszystkie transformacje. Nie było możliwości porównania czasu wykonania z i bez cache.
**
Locality Level**

Nie było możliwości sprawdzenia Locality Level. W teorii najbardziej korzystnym poziomem jest PROCESS_LOCAL, ponieważ dane są przetwarzane bezpośrednio z pamięci procesu.

**Wnioski**

Analiza wykazała, że wszystkie odczyty miały długość 151 nukleotydów. Spośród 2 319 626 odczytów 2 313 942 nie zawierały znaku #, natomiast 5 684 zawierały co najmniej jeden znak #, co wskazuje na niską jakość odczytu.